# Lab 01 — Eval-First: Build the Baseline  (STARTER)

**Section 1 · Foundations** · Advanced RAG Techniques · Helix Medical / HelixRAG

Fill in every cell marked `# TODO`. Setup first: read **README.md**, run `docker compose up -d`, then `pip install -r requirements.txt`. The corpus + golden set come from `helix_corpus.py` (in this same folder). The infrastructure cells (imports, embedder) are given — you implement generation, the three metrics, and the scorecard loop.

In [ ]:
import sys, os, time
for _p in (".", ".."):
    sys.path.insert(0, os.path.abspath(_p))   # find helix_corpus.py
import numpy as np
from helix_corpus import CORPUS, GOLDEN, DOC_IDS, TEXTS, SOURCE_TYPE, BASELINE, TARGETS

# --- Local models via Ollama (no API key). Start the stack first:  docker compose up -d
GEN_MODEL   = os.environ.get("GEN_MODEL",   "qwen2.5:3b")     # generation
JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "qwen2.5:3b")     # LLM-as-judge (cheap)
SMALL_MODEL = os.environ.get("SMALL_MODEL", "qwen2.5:1.5b")   # cheap tier for cost routing (Lab 6)

from openai import OpenAI
client = OpenAI(base_url=os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
                api_key="ollama")   # Ollama ignores the key; nothing to configure

def llm(prompt, system=None, model=GEN_MODEL, max_tokens=300):
    msgs = [{"role":"user","content":prompt}]
    if system: msgs = [{"role":"system","content":system}] + msgs
    r = client.chat.completions.create(model=model, messages=msgs,
                                       max_tokens=max_tokens, temperature=0)
    return r.choices[0].message.content.strip()


## 1. Naive retriever — one dense search, take the single best match
The simplest possible baseline: embed the query, grab the top-1 chunk. It's what a first-cut RAG demo does, and it's exactly what we're going to beat.

In [ ]:
from sentence_transformers import util   # torch cos_sim helper; reranker also uses sentence-transformers
EMB_MODEL = os.environ.get("EMB_MODEL", "qwen3-embedding:0.6b")

class OllamaEmbedder:
    """Drop-in for SentenceTransformer: .encode(text | list[str]) -> normalized numpy array, via Ollama."""
    def encode(self, texts, normalize_embeddings=True, **_):
        single = isinstance(texts, str)
        batch = [texts] if single else list(texts)
        data = client.embeddings.create(model=EMB_MODEL, input=batch).data
        v = np.array([d.embedding for d in data], dtype="float32")
        if normalize_embeddings:
            v = v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-9)
        return v[0] if single else v

emb = OllamaEmbedder()
doc_vecs = emb.encode(TEXTS)

def dense_retrieve(q, k=3):
    qv = emb.encode(q, normalize_embeddings=True)
    sims = util.cos_sim(qv, doc_vecs)[0]
    top = sims.argsort(descending=True)[:k]
    return [(DOC_IDS[i], TEXTS[i]) for i in top]


## 2 & 3. Generation + the eval metrics
`context_recall` = did the gold doc make the cut. `faithfulness` = grounded in the retrieved text. `answer_correctness` = agrees with the clinician's reference answer. Watch how these diverge — that divergence is the whole lesson.

In [ ]:
import re

# TODO: build a context string from ctx (a list of (doc_id, text)), then call llm(...)
# asking for a ONE-sentence answer that uses ONLY the context and cites its source [i].
def generate(q, ctx, model=GEN_MODEL):
    raise NotImplementedError("TODO: implement generate()")

# TODO: return 1.0 if gold_doc is among the retrieved doc ids, else 0.0
def context_recall(retrieved, gold_doc):
    raise NotImplementedError("TODO: implement context_recall()")

# TODO: LLM-as-judge for grounding. Have the model reason claim-by-claim, then end with
# 'SCORE: x' (0.0-1.0), and parse that trailing number. Measures: is the answer supported by ctx?
def faithfulness(answer, ctx, model=JUDGE_MODEL):
    raise NotImplementedError("TODO: implement faithfulness()")

# TODO: LLM-as-judge for correctness. Compare the answer to the clinician's reference gold_answer;
# end with 'SCORE: x' and parse. This is the metric that exposes a 'faithful but wrong' answer.
def answer_correctness(answer, gold_answer, model=JUDGE_MODEL):
    raise NotImplementedError("TODO: implement answer_correctness()")


## 4. The baseline scorecard
Run the golden set through naive RAG (single best match) and write the numbers down.

In [ ]:
naive_retrieve = lambda q: dense_retrieve(q, 1)   # top-1: the naive baseline (given)

# TODO: run the whole GOLDEN set through naive RAG. For each item: retrieve, generate an answer,
# and collect context_recall, faithfulness, and answer_correctness. Return the mean of each.
def evaluate(retriever=naive_retrieve, gen=generate):
    raise NotImplementedError("TODO: implement the evaluation loop")

scorecard = evaluate()
print("NAIVE BASELINE:", scorecard)
# Deliverable: paste this scorecard, and name the golden question that fails and why.


### The smoking gun
Inspect the warfarin query — dense similarity ranks the short, generic *superseded* guideline first, so the single-best-match answer is built on stale evidence.

In [ ]:
q = "Is it safe to co-prescribe warfarin and amiodarone?"
ctx = naive_retrieve(q)
print("retrieved:", [d for d,_ in ctx])
print("answer   :", generate(q, ctx))
# g-044 (superseded) is the top hit, so the naive answer says "safe" — the dangerous failure.